# SPFC Evaluation Bench - Kaggle T2I Only

Kaggle runbook for SPFC vs Rectified-CFG++ vs base SD3 Medium on the 100-prompt T2I-CompBench subset only. Every generation/evaluation run is isolated in its own cell and streams progress while it runs.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Soobiwan/aim-flow.git'
SEED = 13
RUN_ROOT = '/kaggle/working/spfc_eval_seed13/runs'
REPORT_DIR = '/kaggle/working/spfc_eval_seed13/reports'
EVAL_DIR = f'{REPORT_DIR}/eval'
T2I_MANIFEST = '/kaggle/working/aim-flow/configs/t2i_compbench_100_seed13.json'
T2I_DECOMP = '/kaggle/working/aim-flow/configs/t2i_compbench_100_seed13_spfc.json'
T2I_DATASET_ROOT = '/kaggle/working/aim-flow/external/T2I-CompBench/examples/dataset'
EXECUTE_T2I_OFFICIAL = True
QUALITATIVE_MANIFEST = T2I_MANIFEST

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/aim-flow
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
# If Kaggle gives a P100 with an incompatible Torch build, uncomment these and restart the runtime.
!pip uninstall -y torch torchvision torchaudio
!pip install --no-cache-dir --force-reinstall torch==2.4.1+cu118 torchvision==0.19.1+cu118 --index-url https://download.pytorch.org/whl/cu118
!pip install -e .
!pip install datasets

In [ ]:
# Official T2I-CompBench evaluator dependencies.
!pip install timm==0.4.12 fairscale==0.4.4 ruamel.yaml opencv-python yacs pycocotools spacy ftfy regex
!pip install 'git+https://github.com/facebookresearch/detectron2.git@5aeb252b194b93dc2879b4ac34bc51a31b5aee13'
!python -m spacy download en_core_web_sm


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
from aim_flow.eval_bench.prompt_sources import ensure_t2i_compbench_dataset

ensure_t2i_compbench_dataset(T2I_DATASET_ROOT)
weight_dir = Path('/kaggle/working/aim-flow/external/T2I-CompBench/UniDet_eval/experts/expert_weights')
weight_dir.mkdir(parents=True, exist_ok=True)
weight_path = weight_dir / 'Unified_learned_OCIM_RS200_6x+2x.pth'
if not weight_path.exists():
    urlretrieve('https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth', str(weight_path))
print(f'Official T2I evaluator is ready: {weight_path}')


In [ ]:
import os
from IPython.display import Markdown, display

MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
HF_SECRET_NAMES = ('Huggingface', 'HF_TOKEN', 'HUGGINGFACE_TOKEN')

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in HF_SECRET_NAMES:
            try:
                token = secrets.get_secret(secret_name)
            except Exception:
                token = None
            if token:
                os.environ['HF_TOKEN'] = token
                break
    except Exception:
        pass

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not token:
    raise RuntimeError(
        'Missing Hugging Face token. In Kaggle, add a secret named Huggingface or HF_TOKEN, '
        'turn it on for this notebook, and make sure that Hugging Face account has accepted the SD3 Medium license.'
    )

try:
    from huggingface_hub import HfApi
    HfApi().model_info(MODEL_ID, token=token)
except Exception as exc:
    raise RuntimeError(
        f'HF_TOKEN is set, but access check for {MODEL_ID} failed. '
        'Confirm the Kaggle secret is enabled and the token account has accepted the gated model license.'
    ) from exc

display(Markdown('Hugging Face token is configured and can access SD3 Medium.'))


In [ ]:
import json, subprocess, time
from pathlib import Path
from IPython.display import Image as DisplayImage, Markdown, display

EST_SEC_PER_PROMPT = {
    'spfc_generation': 240,
    'rectified_cfgpp_generation': 90,
    'base_generation': 45,
    't2i_official_eval': 8,
    't2i_stage_only': 0.05,
}

def manifest_count(path, default=100):
    path = Path(path)
    if not path.exists():
        return default
    return len(json.loads(path.read_text(encoding='utf-8'))['samples'])

def fmt_seconds(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}h {m}m {s}s' if h else f'{m}m {s}s'

def timed_run(label, command, estimated_seconds):
    display(Markdown(f'### {label}
Estimated time: **{fmt_seconds(estimated_seconds)}**'))
    start = time.perf_counter()
    result = subprocess.run(command, shell=True, text=True)
    elapsed = time.perf_counter() - start
    display(Markdown(f'Finished **{label}** in **{fmt_seconds(elapsed)}**.'))
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')

def show_scores(path):
    path = Path(path)
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        display(Markdown('```json
' + json.dumps(data.get('scores', data), indent=2) + '
```'))

def show_markdown(path):
    path = Path(path)
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))

def show_image(path):
    path = Path(path)
    if path.exists():
        display(DisplayImage(filename=str(path)))

In [ ]:
# timed_run('Prepare T2I-CompBench 100-prompt manifest and SPFC template', f'python scripts/bench_prepare_prompts.py --benchmark t2i_compbench --seed {SEED} --t2i-subset-size 100 --t2i-dataset-root {T2I_DATASET_ROOT} --write-decomposition-template', 5)

Replace the generated decomposition templates with real LLM/manual SPFC decompositions before generation.

In [ ]:
# timed_run('Validate T2I SPFC decompositions', f'python scripts/bench_validate_decompositions.py --manifest {T2I_MANIFEST} --decompositions {T2I_DECOMP}', 2)

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('SPFC T2I-CompBench generation', f'python scripts/bench_generate.py --manifest {T2I_MANIFEST} --decompositions {T2I_DECOMP} --run-root {RUN_ROOT} --methods spfc --seed {SEED}', N * EST_SEC_PER_PROMPT['spfc_generation'])

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('Rectified-CFG++ T2I-CompBench generation', f'python scripts/bench_generate.py --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --methods rectified_cfgpp --seed {SEED}', N * EST_SEC_PER_PROMPT['rectified_cfgpp_generation'])

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('Base SD3 T2I-CompBench generation', f'python scripts/bench_generate.py --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --methods base --seed {SEED}', N * EST_SEC_PER_PROMPT['base_generation'])

In [ ]:
score_path = Path(EVAL_DIR) / 't2i_compbench_scores.json'
if score_path.exists():
    score_path.unlink()

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('SPFC T2I-CompBench evaluation', f'python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods spfc --append {flag}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('Rectified-CFG++ T2I-CompBench evaluation', f'python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods rectified_cfgpp --append {flag}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('Base SD3 T2I-CompBench evaluation', f'python scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods base --append {flag}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
T2I_SCORES = f'{EVAL_DIR}/t2i_compbench_scores.json'
QUAL_GRID = f'{REPORT_DIR}/qualitative_grid.png'
timed_run('Build T2I table and qualitative grid', f'python scripts/bench_report.py --t2i-scores {T2I_SCORES} --run-root {RUN_ROOT} --output-dir {REPORT_DIR} --qualitative-manifest {QUALITATIVE_MANIFEST} --qualitative-output {QUAL_GRID}', 10)
show_markdown(Path(REPORT_DIR) / 't2i_compbench_table.md')
show_image(QUAL_GRID)